## Import Library and Variables

In [ ]:
import os
from pathlib import Path

while not Path("configs/config.yaml").exists():
  parent = os.path.dirname(os.getcwd())
  if parent == os.getcwd():
      raise RuntimeError("Root project tidak ketemu.")
  os.chdir(parent)

import yaml
import tensorflow as tf

CONFIG  = yaml.safe_load(Path("configs/config.yaml").read_text())
SPLITS   = Path(CONFIG["data"]["splits_dir"]) 
CLASSES  = CONFIG["data"]["classes"]
IMG_SIZE = CONFIG["image"]["size"]              
BATCH    = CONFIG["train"]["batch_size"]        
SEED     = CONFIG["seed"]                        

print("TF version:", tf.__version__)

TF version: 2.21.0


## Load Data

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
  SPLITS / "train",
  labels="inferred",          
  label_mode="int",          
  class_names=CLASSES,      
  image_size=(IMG_SIZE, IMG_SIZE),
  batch_size=BATCH,
  shuffle=True,
  seed=SEED,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
  SPLITS / "val",
  labels="inferred", label_mode="int", class_names=CLASSES,
  image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH, shuffle=False,
)

test_ds = tf.keras.utils.image_dataset_from_directory(
  SPLITS / "test",
  labels="inferred", label_mode="int", class_names=CLASSES,
  image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH, shuffle=False,
)

print(train_ds.class_names)

Found 3062 files belonging to 4 classes.
Found 381 files belonging to 4 classes.
Found 386 files belonging to 4 classes.
['angry', 'happy', 'relaxed', 'sad']


## Train Model

In [5]:
from tensorflow.keras import layers

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(AUTOTUNE)
val_ds   = val_ds.cache().prefetch(AUTOTUNE)
test_ds  = test_ds.cache().prefetch(AUTOTUNE)

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

In [6]:
num_classes = len(CLASSES)

model_cnn = tf.keras.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    data_augmentation,
    layers.Rescaling(1./255),        

    layers.Conv2D(32, 3, activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(128, 3, activation="relu"),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dropout(0.5),
    layers.Dense(128, activation="relu"),
    layers.Dense(num_classes),           
])

model_cnn.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,476 (42.61 MB)

 Trainable params: 11,169,476 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model_cnn.compile(
    optimizer=tf.keras.optimizers.Adam(CONFIG["train"]["lr"]),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)

callbacks = [
  tf.keras.callbacks.EarlyStopping(
    patience=CONFIG["train"]["early_stopping_patience"],
    restore_best_weights=True),
  tf.keras.callbacks.ModelCheckpoint(
    "models/cnn.keras", save_best_only=True),
]

history = model_cnn.fit(
    train_ds,
    validation_data=val_ds,
    epochs=CONFIG["train"]["epochs"],
    callbacks=callbacks,
)

c:\Dev\Project UAS ML Prak\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


Epoch 1/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 258s 2s/step - accuracy: 0.2717 - loss: 1.3959 - val_accuracy: 0.3228 - val_loss: 1.3088
Epoch 2/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 283s 3s/step - accuracy: 0.3416 - loss: 1.3446 - val_accuracy: 0.3648 - val_loss: 1.3544
Epoch 3/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 232s 2s/step - accuracy: 0.3703 - loss: 1.3117 - val_accuracy: 0.4173 - val_loss: 1.2314
Epoch 4/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 167s 2s/step - accuracy: 0.4141 - loss: 1.2797 - val_accuracy: 0.4278 - val_loss: 1.2702
Epoch 5/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 153s 2s/step - accuracy: 0.4438 - loss: 1.2270 - val_accuracy: 0.4882 - val_loss: 1.1690
Epoch 6/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.4595 - loss: 1.1882

In [ ]:
import matplotlib.pyplot as plt

acc = history.history["accuracy"];      val_acc = history.history["val_accuracy"]
loss = history.history["loss"];         val_loss = history.history["val_loss"]

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(acc, label="Train"); plt.plot(val_acc, label="Validation")
plt.title("Model Accuracy"); plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend()

plt.subplot(1, 2, 2)
plt.plot(loss, label="Train"); plt.plot(val_loss, label="Validation")
plt.title("Model Loss"); plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend()
plt.savefig("reports/figures/cnn_training.png", dpi=120, bbox_inches="tight")
plt.show()

## Prediction Model

In [ ]:
import numpy as np

y_true = np.concatenate([y for x, y in test_ds], axis=0)   
y_pred_logits = model_cnn.predict(test_ds)                 
y_pred = np.argmax(y_pred_logits, axis=1)                 

print("jumlah test:", len(y_true))
print("contoh y_true:", y_true[:10])
print("contoh y_pred:", y_pred[:10])

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_true, y_pred, target_names=CLASSES, digits=3))

## Metrik Evaluasi

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

ConfusionMatrixDisplay.from_predictions(
    y_true, y_pred, display_labels=CLASSES, cmap="Blues", colorbar=True
)
plt.title("Confusion Matrix - CNN Baseline")
plt.savefig("reports/figures/cnn_confusion.png", dpi=120, bbox_inches="tight")
plt.show()